# House Price Prediction


### 1. Problem Statement

- House prices in Bengaluru vary widely with location, area, and size, making it hard to judge if a property is fairly priced.
- Buyers and sellers need a reliable, data-driven way to estimate prices instead of relying on guesswork.
- Goal: Build a machine learning regression model to predict house price (in lakhs) from location, total sqft, BHK, and number of bathrooms.


### 2. Load Dataset and Import Required Packages


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('../data/src/housing.csv')
df.head()


#### Shape of Dataset


In [ ]:
print(f"Rows of Dataset are {df.shape[0]} and Columns of Dataset are {df.shape[1]}.")


#### Dataset Information

- 13,320 rows and 9 columns
- Features: area_type, availability, location, size, society, total_sqft, bath, balcony
- Target: price (in lakhs)
- Missing values in society, balcony, bath, size, and location
- Duplicate rows are present


#### Data Checks to Perform

- Check missing values
- Check duplicates
- Check data types
- Check number of unique values in each column
- Check dataset statistics
- Check categories in categorical columns


##### Check Missing Values


In [ ]:
df.isnull().sum()


In [ ]:
df['location'] = df['location'].fillna(df['location'].mode()[0])
df['size'] = df['size'].fillna(df['size'].mode()[0])
df['bath'] = df['bath'].fillna(df['bath'].mean())
df['balcony'] = df['balcony'].fillna(df['balcony'].median())


In [ ]:
df.drop(columns=['area_type', 'availability', 'society'], inplace=True)
df.isnull().sum()


#### Check Duplicates


In [ ]:
df.duplicated().sum()


In [ ]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()


#### Check Data Type


In [ ]:
df.info()


#### Check the Number of Unique Values of Each Column


In [ ]:
df.nunique()


##### Check Dataset Statistics


In [ ]:
df.describe()


In [ ]:
df.location.value_counts()


In [ ]:
df['location'] = df['location'].apply(lambda x: x.strip())
df['location']


In [ ]:
location_stats = df.groupby('location')['location'].agg('count').sort_values(ascending=True)
location_stats


In [ ]:
location_less_than_ten_entries = location_stats[location_stats <= 10]
location_less_than_ten_entries


In [ ]:
df['location'] = df['location'].apply(
    lambda x: 'other' if x in location_less_than_ten_entries else x
)
df['location'].value_counts()


In [ ]:
df['size'].unique()


In [ ]:
df['bedrooms'] = df['size'].apply(lambda x: int(x.split(' ')[0]))
df.head()


In [ ]:
df.total_sqft.unique()


In [ ]:
def clean_sqft(sqft):
    tokens = str(sqft).split('-')
    if len(tokens) == 2:
        try:
            return (float(tokens[0]) + float(tokens[1])) / 2
        except:
            return None
    try:
        return float(sqft)
    except:
        return None

print(clean_sqft('300-500'))


In [ ]:
df['total_sqft'] = df['total_sqft'].apply(clean_sqft)
df['total_sqft'].unique()


In [ ]:
df.head()


In [ ]:
df.describe()


In [ ]:
df['sqft_per_bed'] = df['total_sqft'] / df['bedrooms']


In [ ]:
df.sqft_per_bed.describe()


In [ ]:
df2 = df[df['sqft_per_bed'] >= 300].copy()
df2.head()


In [ ]:
df2['price_per_sqft'] = round(
    df2['price'] * 100000 / df2['total_sqft'], 2
)
df2.price_per_sqft.describe()


In [ ]:
df3 = df2[df2['price_per_sqft'] >= 2000].copy()
df3.head()


In [ ]:
df3.drop(
    columns=['size', 'sqft_per_bed', 'price_per_sqft'],
    inplace=True
)
df3.head()


### Model Training and Preprocessing


In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer


In [ ]:
col_trans = make_column_transformer(
    (OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['location']),
    remainder='passthrough'
)


In [ ]:
lr = LinearRegression()


In [ ]:
scaler = StandardScaler()


In [ ]:
model = make_pipeline(col_trans, scaler, lr)


In [ ]:
X = df3.drop(columns=['price'])
y = df3['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
model.fit(X_train, y_train)


In [ ]:
model.score(X_test, y_test)


### Example Prediction


In [ ]:
input_data = pd.DataFrame(
    [['Electronic City Phase II', 2000.0, 2.0, 1.0, 3]],
    columns=['location', 'total_sqft', 'bath', 'balcony', 'bedrooms']
)

model.predict(input_data)


### Save Model


In [ ]:
import pickle as pk

model_path = '../models/House_prediction_model.pkl'
with open(model_path, 'wb') as file:
    pk.dump(model, file)

print("Model saved successfully:", model_path)


### Save Cleaned Dataset


In [ ]:
cleaned_data_path = '../data/Cleaned_data.csv'
df3.to_csv(cleaned_data_path, index=False)

print("Cleaned dataset saved successfully:", cleaned_data_path)
